# NFL Draft classification project

**Research question:** How accurately can draft position and player characteristics distinguish late-round NFL Draft selections with at least one recorded Pro Bowl selection from those with none?

**Run All is safe for development:** this notebook reads only `Data/development_set.csv`. It does not recreate the split or open the reporting CSV. The original description below is a fixed record of descriptive work performed before reservation; subsequent calculated tables are labeled development-only.

**Original dataset description**

The NFL Draft 1985–2015 dataset, published by Ulrik Thyge Pedersen and downloaded from Kaggle [1], contains 8,435 draft-selection records and 34 variables covering draft details, player characteristics, and NFL performance. Among the 7,190 records with age reported, mean recorded age was 22.64 years (SD = 0.84), median age was 23 years (interquartile range: 22–23), and ages ranged from 20 to 29 years. Of records with known age, 7.1% were aged 20–21, 81.2% were aged 22–23, and 11.8% were aged 24–29; age was missing for 1,245 records (14.8% of the full dataset). The dataset includes 15 standardized position categories, with defensive backs (18.6%), linebackers (13.3%), and wide receivers (12.4%) the most common. College information is available for 5,005 records (59.3%), representing 317 distinct recorded college labels. Sex, race, and ethnicity are not recorded and cannot be summarized from this file.

**IEEE reference**

[1] U. T. Pedersen, “NFL Draft 1985–2015,” Kaggle, data set. Accessed: Nov. 17, 2026. [Online]. Available: https://www.kaggle.com/datasets/ulrikthygepedersen/nfl-draft-1985-2015

## Load development data

Locate the repository folder and read the 80% development CSV. The original raw CSV and frozen reporting CSV must not be used for further exploration.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd() / "group-04-final-project", Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "Data/development_set.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the group-04-final-project folder.")
df = pd.read_csv(ROOT / "Data/development_set.csv")
print(f"Development only: {len(df):,} records and {df.shape[1]} original variables.")
display(df.head(3))

Development only: 6,748 records and 34 original variables.


,column_a,player_id,year,rnd,pick,tm,player,hof,pos,position_standard,first4av,age,to,ap1,pb,st,carav,drav,g,cmp,pass_att,pass_yds,pass_td,pass_int,rush_att,rush_yds,rush_tds,rec,rec_yds,rec_tds,tkl,def_int,sk,college_univ
0,2015Jameis Winston,WinsJa00,2015,1,1,TAM,Jameis Winston,No,QB,QB,13,21.0,2016.0,0,1,2,13.0,13.0,26.0,540.0,913.0,6722.0,42.0,25.0,86.0,311.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,Florida St.
1,2015Marcus Mariota,MariMa01,2015,1,2,TEN,Marcus Mariota,No,QB,QB,9,21.0,2016.0,0,0,2,9.0,9.0,23.0,458.0,725.0,5590.0,42.0,18.0,79.0,516.0,4.0,1.0,41.0,1.0,NaN,NaN,NaN,Oregon
2,2015Amari Cooper,CoopAm00,2015,1,4,OAK,Amari Cooper,No,WR,WR,9,21.0,2016.0,0,1,1,9.0,9.0,26.0,NaN,NaN,NaN,NaN,NaN,3.0,-3.0,0.0,134.0,1970.0,9.0,NaN,NaN,NaN,Alabama


## Age and other player characteristics — development only

Sample SD uses `ddof=1`. Age-band percentages use observed ages only. Position and college availability use all development rows. No demographic values are imputed; no sex or ethnicity is inferred.

In [2]:
ages = pd.to_numeric(df["age"], errors="raise")
known = ages.dropna()
age_summary = pd.Series({
    "Development records": len(df), "Age observed": len(known),
    "Age missing": ages.isna().sum(), "Age missing (%)": 100 * ages.isna().mean(),
    "Mean age": known.mean(), "Sample SD": known.std(ddof=1),
    "Median": known.median(), "25th percentile": known.quantile(.25),
    "75th percentile": known.quantile(.75), "Minimum": known.min(), "Maximum": known.max()
}, name="Development only")
display(age_summary.to_frame().round(2))
bands = pd.cut(known, bins=[19,21,23,29], labels=["20–21","22–23","24–29"])
age_bands = bands.value_counts(sort=False).rename("Count").to_frame()
age_bands["Percent of known ages"] = (100 * age_bands["Count"] / len(known)).round(1)
display(age_bands)
positions = df["position_standard"].fillna("Not recorded").value_counts().rename("Count").to_frame()
positions["Percent of development records"] = (100 * positions["Count"] / len(df)).round(1)
display(positions)
print(f"College recorded: {df.college_univ.notna().sum():,} ({100*df.college_univ.notna().mean():.1f}%).")
print(f"Distinct recorded college labels: {df.college_univ.nunique()}.")
print("Sex, race, and ethnicity: unavailable in this dataset.")

College recorded: 4,015 (59.5%).
Distinct recorded college labels: 289.
Sex, race, and ethnicity: unavailable in this dataset.


,Development only
Development records,6748.00
Age observed,5753.00
Age missing,995.00
Age missing (%),14.75
Mean age,22.64
Sample SD,0.86
Median,23.00
25th percentile,22.00
75th percentile,23.00
Minimum,20.00


,Count,Percent of known ages
age,,
20–21,422,7.3
22–23,4631,80.5
24–29,700,12.2


,Count,Percent of development records
position_standard,,
DB,1249,18.5
LB,884,13.1
WR,838,12.4
RB,652,9.7
DE,575,8.5
DT,520,7.7
T,495,7.3
G,431,6.4
TE,406,6.0


## Predeclared classification cohort — development only

Use 1994–2010, rounds 1–7, to keep the seven-round era and give all classes at least six completed calendar seasons through 2015. This does not make career outcomes complete. `success=1` means at least one recorded Pro Bowl; `success=0` means none recorded. This cell creates the development target, but does not train a model or access reporting data.

Late-round performance (rounds 4–7) is primary. Train on all eligible rounds; use early rounds (1–3) for a secondary evaluation comparison. Apply the same eligibility rule to the reporting partition only in the final phase.

In [3]:
eligible = df["year"].between(1994, 2010) & df["rnd"].between(1, 7)
model_df = df.loc[eligible].copy()
y = model_df["pb"].ge(1).astype(int).rename("success")
predictors = ["pick", "age", "position_standard", "college_univ", "tm", "year"]
X = model_df[predictors].copy()
groups = model_df["player_id"].fillna(
    "selection:" + model_df["year"].astype(str) + ":" + model_df["pick"].astype(str)
)
print(f"Eligible development records: {len(model_df):,}")
print(f"Eligible late-round development records: {model_df.rnd.between(4,7).sum():,}")
print(f"Development positives: {y.sum():,}; development positive rate: {y.mean():.1%}")
print("X contains draft-time predictors only; y and groups are separate.")

Eligible development records: 3,414
Eligible late-round development records: 2,086
Development positives: 399; development positive rate: 11.7%
X contains draft-time predictors only; y and groups are separate.


## Next steps

Use grouped, stratified cross-validation on `X`, `y`, and `groups`. Keep all learned preprocessing inside the training folds. Compare a majority baseline, a pick-only logistic regression, multivariable logistic regression, and random forest. Select model settings and the classification threshold using development validation only. Report average precision, precision, recall, F1, balanced accuracy, and calibration; ordinary accuracy alone is insufficient for rare successes.

Freeze the final model and decision threshold before any reporting-set evaluation. Do not add reporting-data loading or evaluation cells until the final phase. The README documents the split, limits of the outcome, and prior full-dataset exposure.